In [6]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from time import sleep
import re
import io

# PDF 텍스트 추출용 (PDF도 쓰고 싶으면 필요)
# pip install pdfplumber
import pdfplumber


# 1. 기존 성명문 엑셀 불러오기
# ---------------------------------------------------
excel_path = "fomc_statement.xlsx"   # 실제 경로로 수정
df = pd.read_excel(excel_path)

df = df.drop_duplicates(subset=["meeting_id", "doc_type"])
print(df.head())


# 2. meeting_id 에서 YYYYMMDD 추출 (형식 여러 개 대응)
# ---------------------------------------------------
def extract_date_str(meeting_id: str) -> str:
    """
    meeting_id 예시:
      - 'FOMC_20070918'
      - 'FOMC20070918'
      - '2007-09-18'
      - '20070918'
    등을 최대한 유연하게 처리해서 '20070918' 로 반환
    """
    # 숫자 8자리만 뽑기
    digits = re.findall(r"\d{8}", meeting_id)
    if digits:
        return digits[0]

    # 숫자4-숫자2-숫자2 패턴이면 합치기
    m = re.search(r"(\d{4})[-_/\.](\d{2})[-_/\.](\d{2})", meeting_id)
    if m:
        return "".join(m.groups())

    # 못 찾으면 일단 원본 리턴 (필요 시 직접 수정)
    return meeting_id


# 3. 의사록 URL 후보들을 만드는 함수
#    (연도/시기에 따라 위치가 달라서 여러 개 시도)
# ---------------------------------------------------
def build_minutes_candidate_urls(date_str: str):
    """
    실제로 쓰이는 대표적인 패턴들:

    1) 예전 연도 (2000s 등)
       https://www.federalreserve.gov/fomc/minutes/20070918.htm

    2) 최근 연도 (2015, 2019 등)
       https://www.federalreserve.gov/monetarypolicy/fomcminutes20151216.htm
       https://www.federalreserve.gov/monetarypolicy/fomcminutes20190320.htm

    3) PDF 버전
       https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20151216.pdf
       https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20190320.pdf
    """
    base = "https://www.federalreserve.gov"

    candidates = [
        f"{base}/monetarypolicy/fomcminutes{date_str}.htm",
        f"{base}/monetarypolicy/fomcminutes{date_str}",          # 종종 .htm 없이도 동작
        f"{base}/monetarypolicy/files/fomcminutes{date_str}.pdf",
        f"{base}/fomc/minutes/{date_str}.htm",
    ]
    return candidates


# 4. HTML minutes 에서 텍스트 추출
# ---------------------------------------------------
def parse_minutes_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    candidates = []

    # 대표적인 main 영역 id 몇 개 먼저 시도
    for div_id in ["article", "content", "col1", "main"]:
        div = soup.find("div", id=div_id)
        if div:
            ps = div.find_all("p")
            if ps:
                text = "\n\n".join(p.get_text(strip=True) for p in ps)
                candidates.append(text)

    # 못 찾으면 그냥 전체 p 태그 모으기
    if not candidates:
        ps = soup.find_all("p")
        if ps:
            text = "\n\n".join(p.get_text(strip=True) for p in ps)
            candidates.append(text)

    # 그래도 없으면 통째로 텍스트
    if not candidates:
        return soup.get_text("\n", strip=True)

    # 제일 긴 후보(대개 본문)를 사용
    return max(candidates, key=len)


# 5. PDF minutes 에서 텍스트 추출
# ---------------------------------------------------
def parse_minutes_pdf(content: bytes) -> str:
    text_pages = []
    with pdfplumber.open(io.BytesIO(content)) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            text_pages.append(text)
    return "\n\n".join(text_pages)


# 6. 의사록 크롤링 루프
# ---------------------------------------------------
minutes_rows = []

existing_minutes_ids = set(
    df.loc[df["doc_type"] == "minutes", "meeting_id"].tolist()
) if "minutes" in df["doc_type"].unique() else set()

unique_meetings = df["meeting_id"].unique()

for meeting_id in unique_meetings:
    if meeting_id in existing_minutes_ids:
        continue

    date_str = extract_date_str(meeting_id)
    candidate_urls = build_minutes_candidate_urls(date_str)

    print(f"\n[INFO] {meeting_id} → try candidates:")
    found = False
    minutes_text = None
    used_url = None

    for url in candidate_urls:
        print(f"  [TRY] {url}")
        try:
            resp = requests.get(url, timeout=20)
        except Exception as e:
            print(f"    [ERROR] request failed: {e}")
            continue

        # 404면 다음 후보
        if resp.status_code == 404:
            print("    [WARN] 404 Not Found")
            continue

        # 200 아니면 일단 스킵
        if resp.status_code != 200:
            print(f"    [WARN] HTTP {resp.status_code} (skip)")
            continue

        # 여기까지 오면 뭔가 찾은 것
        used_url = url

        # PDF인지 HTML인지 content-type 으로 판단
        ctype = resp.headers.get("Content-Type", "")
        try:
            if "pdf" in ctype.lower() or url.lower().endswith(".pdf"):
                print("    [INFO] Detected PDF, parsing as PDF")
                minutes_text = parse_minutes_pdf(resp.content)
            else:
                print("    [INFO] Detected HTML, parsing as HTML")
                minutes_text = parse_minutes_html(resp.text)
        except Exception as e:
            print(f"    [ERROR] parsing failed: {e}")
            minutes_text = None

        if minutes_text and minutes_text.strip():
            found = True
            break
        else:
            print("    [WARN] parsed text empty, try next candidate")

    if not found:
        print(f"  [FAIL] No minutes found for {meeting_id}")
        continue

    row = {
        "meeting_id": meeting_id,
        "doc_type": "minutes",
        "source_url": used_url,
        "text": minutes_text,
    }
    minutes_rows.append(row)

    # 너무 빠른 요청 방지
    sleep(1)

# 7. DataFrame 병합 및 저장
# ---------------------------------------------------
df_minutes = pd.DataFrame(minutes_rows)
print("\n[INFO] 새로 수집된 minutes 개수:", len(df_minutes))

if not df_minutes.empty:
    df_merged = pd.concat([df, df_minutes], ignore_index=True)
else:
    df_merged = df.copy()

# 보기 좋게 meeting_id + doc_type 순으로 정렬
doc_type_order = {"statement": 0, "minutes": 1}
df_merged["doc_type_order"] = df_merged["doc_type"].map(doc_type_order).fillna(99)
df_merged = (
    df_merged.sort_values(["meeting_id", "doc_type_order"])
             .drop(columns=["doc_type_order"])
)

output_path = "fomc_statement_minutes.xlsx"
df_merged.to_excel(output_path, index=False)
print(f"[DONE] 저장 완료: {output_path}")


      meeting_id   doc_type  \
0  FOMC_20060131  statement   
1  FOMC_20060328  statement   
2  FOMC_20060510  statement   
3  FOMC_20060629  statement   
4  FOMC_20060808  statement   

                                          source_url  \
0  https://www.federalreserve.gov/newsevents/pres...   
1  https://www.federalreserve.gov/newsevents/pres...   
2  https://www.federalreserve.gov/newsevents/pres...   
3  https://www.federalreserve.gov/newsevents/pres...   
4  https://www.federalreserve.gov/newsevents/pres...   

                                                text  
0  January 31, 2006\n\nFor immediate release\n\nT...  
1  March 28, 2006\n\nFor immediate release\n\nThe...  
2  May 10, 2006\n\nFor immediate release\n\nThe F...  
3  June 29, 2006\n\nFor immediate release\n\nThe ...  
4  August 08, 2006\n\nFor immediate release\n\nTh...  

[INFO] FOMC_20060131 → try candidates:
  [TRY] https://www.federalreserve.gov/monetarypolicy/fomcminutes20060131.htm
    [WARN] 404 Not Found
  [

In [8]:
import yfinance as yf
df = yf.download("SPY", start="2006-01-01")
df.to_csv("SPY.csv")


/var/folders/wf/5q4y6xhj0tg49_pzxr2tnd600000gn/T/ipykernel_11038/2089817430.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("SPY", start="2006-01-01")
[*********************100%***********************]  1 of 1 completed


In [10]:
yf.download("ZQ=F")
df.to_csv('ZQ.csv')

/var/folders/wf/5q4y6xhj0tg49_pzxr2tnd600000gn/T/ipykernel_11038/1270461638.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  yf.download("ZQ=F")
[*********************100%***********************]  1 of 1 completed
